# Single-cell Drug-Target Nomination (Identification)
***Artificial Intelligence For Science - 4960*** <br>
***Task Five - Traditional AI: Support Vector Regression, Random Forests, Gradient Boosting*** <br>
***By: Swaroop Sridhar*** <br>



### Import Libraries
Here we import the Python libraries needed for this ML task:
- Core data science: NumPy, Pandas
- Visualization: Matplotlib, Seaborn
- TDC DataLoader for drug-target interaction data
- Scikit-learn utilities for preprocessing and metrics
- Three different ML models: CatBoost, RandomForest, and Logistic Regression
- Various evaluation metrics for model assessment

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tdc.resource.dataloader import DataLoader
from sklearn.preprocessing import StandardScaler, LabelEncoder
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_recall_curve, roc_curve, auc
from sklearn.metrics import classification_report, confusion_matrix, balanced_accuracy_score
from sklearn.utils.class_weight import compute_class_weight

TypeError: Can't instantiate abstract class Path with abstract methods __truediv__, read_bytes, read_text

### Data Loading and Preprocessing Function
Here we define a function that:
- Loads the OpenTargets drug-target interaction dataset
- Encodes categorical variables (protein and cell type) using LabelEncoder
- Implements a cold-split strategy for proteins to ensure no protein overlap between splits
- Creates train/validation/test masks for data separation

This approach ensures a more realistic evaluation of the model's generalization ability.

In [ ]:
def load_and_preprocess_data():
    # Load dataset
    data = DataLoader(name="opentargets_dti")
    df = data.get_data()
    
    # Encode categorical variables
    le_protein = LabelEncoder()
    le_cell = LabelEncoder()
    
    df['protein_encoded'] = le_protein.fit_transform(df['protein'])
    df['cell_encoded'] = le_cell.fit_transform(df['cell_type'])
    
    # Split proteins for cold-split
    unique_proteins = df['protein'].unique()
    train_proteins, temp_proteins = train_test_split(unique_proteins, test_size=0.2, random_state=42)
    val_proteins, test_proteins = train_test_split(temp_proteins, test_size=0.5, random_state=42)
    
    # Create dataset splits
    train_mask = df['protein'].isin(train_proteins)
    val_mask = df['protein'].isin(val_proteins)
    test_mask = df['protein'].isin(test_proteins)
    
    return df, train_mask, val_mask, test_mask

### Feature Engineering Function
Here we define a function that creates features from the raw data:
- Converts encoded protein and cell type IDs into features
- Optionally includes protein length if sequence data is available
- Creates binary features for disease types (RA and IBD)

The function transforms raw biological data into a format suitable for machine learning models.

In [ ]:
def create_features(df):
    """Create features from protein and cell type data"""
    features = pd.DataFrame({
        'protein_id': df['protein_encoded'],
        'cell_type_id': df['cell_encoded']
    })
    
    # Add protein length as feature (if available)
    if 'protein_sequence' in df.columns:
        features['protein_length'] = df['protein_sequence'].str.len()
    
    # Add disease type as feature
    features['is_RA'] = df['disease_type'] == 'RA'
    features['is_IBD'] = df['disease_type'] == 'IBD'
    
    return features


### Model Training and Evaluation Function
Here we implement the training and evaluation pipeline that:
- Handles class imbalance using SMOTE if specified
- Supports different model types with specific training requirements
- Calculates various performance metrics:
  - Standard and balanced accuracy
  - ROC curves and AUC
  - Precision-Recall curves and AUC

Returns a dictionary of evaluation metrics and curves for visualization.

In [ ]:
def train_and_evaluate(model, X_train, X_val, X_test, y_train, y_val, y_test, 
                      use_smote=False, model_name=''):
    """Train model and evaluate performance"""
    
    if use_smote:
        smote = SMOTE(random_state=42)
        X_train_res, y_train_res = smote.fit_resample(X_train, y_train)
    else:
        X_train_res, y_train_res = X_train, y_train
    
    # Train model
    if model_name == 'catboost':
        model.fit(X_train_res, y_train_res,
                 eval_set=(X_val, y_val),
                 verbose=False)
    else:
        model.fit(X_train_res, y_train_res)
    
    # Predictions
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    
    # Calculate metrics
    results = {
        'accuracy': accuracy_score(y_test, y_pred),
        'balanced_accuracy': balanced_accuracy_score(y_test, y_pred),
        'predictions': y_pred,
        'probabilities': y_pred_proba
    }
    
    # ROC curve
    fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
    results['roc_auc'] = auc(fpr, tpr)
    results['fpr'] = fpr
    results['tpr'] = tpr
    
    # PR curve
    precision, recall, _ = precision_recall_curve(y_test, y_pred_proba)
    results['pr_auc'] = auc(recall, precision)
    results['precision'] = precision
    results['recall'] = recall
    
    return results

### Data Processing and Model Training
- Processes the data using previously defined functions
- Scales features using StandardScaler
- Initializes three different models with specific configurations:
  - CatBoost with class weighting
  - Random Forest with balanced classes
  - XGBoost with SMOTE
- Trains all models and collects their performance metrics
Each model uses different strategies to handle class imbalance.

In [ ]:
# Load and preprocess data
df, train_mask, val_mask, test_mask = load_and_preprocess_data()
features = create_features(df)

# Split features and target
X = features.values
y = df['label'].values

X_train = X[train_mask]
X_val = X[val_mask]
X_test = X[test_mask]
y_train = y[train_mask]
y_val = y[val_mask]
y_test = y[test_mask]

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# Initialize models
catboost_model = CatBoostClassifier(
    iterations=500,
    learning_rate=0.1,
    depth=6,
    class_weights={0: 1, 1: 10},  # Handle class imbalance
    random_state=42
)

rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    class_weight='balanced',
    random_state=42
)

xgb_model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=6,
    scale_pos_weight=10,  # Handle class imbalance
    random_state=42
)

# Train and evaluate models
models = {
    'CatBoost': (catboost_model, False),
    'Balanced RF': (rf_model, False),
    'XGBoost+SMOTE': (xgb_model, True)
}

results = {}
for name, (model, use_smote) in models.items():
    results[name] = train_and_evaluate(
        model, X_train_scaled, X_val_scaled, X_test_scaled,
        y_train, y_val, y_test, use_smote, name
    )

### Visualization and Results Analysis
Finally, we create comprehensive visualizations of model performance:
- Plots ROC curves for all models showing true/false positive trade-offs
- Plots Precision-Recall curves highlighting performance on imbalanced data
- Prints detailed classification reports for each model

These visualizations help in comparing model performance across different metrics and identifying the best model for the task.

In [ ]:
def plot_performance_curves(results):
    # ROC curves
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    for name, res in results.items():
        plt.plot(res['fpr'], res['tpr'], 
                label=f'{name} (AUC = {res["roc_auc"]:.3f})')
    plt.plot([0, 1], [0, 1], 'k--')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC Curves')
    plt.legend()

    # PR curves
    plt.subplot(1, 2, 2)
    for name, res in results.items():
        plt.plot(res['recall'], res['precision'],
                label=f'{name} (AUC = {res["pr_auc"]:.3f})')
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.title('Precision-Recall Curves')
    plt.legend()
    plt.tight_layout()
    plt.show()

# Plot results
plot_performance_curves(results)

# Print classification reports
for name, res in results.items():
    print(f"\n{name} Classification Report:")
    print(classification_report(y_test, res['predictions']))